# Capstone, a delegated migration

**Scenario:** three zones of a fulfilment centre move their robots to the next firmware build
overnight. Every zone writes an aisle by aisle log, and a commander decides whether to continue.

The other two sub-modules kept the logs away from the parent. This one adds the second half. What
crosses back has to be a shape the parent can act on, not prose it has to read.

Think of it as a customs declaration. You fill in the boxes, and anything else is refused at the
counter.

## Mechanics

Two different things are declared here.

| Piece | What it declares | What happens when it is broken |
|---|---|---|
| `response_format` with `json_schema` | the reply comes back in the shape you named | the provider refuses the shape, not your code |
| a Pydantic model | the fields, their types, and which are required | `ValidationError`, at the boundary, before the parent |
| `Send` payload | everything the worker can see | nothing, this is the wall from the last lesson |
| the parent state channel | everything the worker can leave behind | nothing can be left behind that has no channel |
| a fallback report | what the parent gets when validation rejects the reply | the parent is handed a flagged report, never a string |

The last row is the contract. A boundary that can return either a report or prose has declared
nothing, because the caller still has to guess.

## The picture

![Heavy logs stay with the workers, a typed report crosses to the commander](images/typed-result-contract.svg)

The counter in the middle takes a filled in form or refuses. Prose never passes through.

## The cost

```
parent_tokens = tokens per report x zones, not tokens per log x zones
```

Reading is paid once per zone either way. The commander's context is paid every turn of the night.

## The failure

Three zones, and a log with the answer buried in the last two lines.

In [1]:
def zone_log(zone, moved, failed):
    """One zone's migration log, the way the fleet controller writes it."""
    lines = [f"ZONE {zone} migration firmware v4.8.1 -> v5.0.2"]
    for aisle in range(1, 23):
        lines.append(f"  aisle {aisle:02d} beacon_sync ok drift=3ms battery=0.82 dock=free")
        lines.append(f"  aisle {aisle:02d} pick_arm calib pass torque=11.4Nm cycles=48210")
    lines.append(f"  result: {moved} robots on v5.0.2, {failed} rolled back after watchdog reset")
    lines.append("  note: charger 3 offline for 11 minutes during the window")
    return "\n".join(lines)


ZONES = {"Z-NORTH": zone_log("Z-NORTH", 118, 2), "Z-SOUTH": zone_log("Z-SOUTH", 96, 0),
         "Z-DOCK": zone_log("Z-DOCK", 41, 9)}
print(f"{len(ZONES)} zones, {len(ZONES['Z-DOCK'])} characters of log each")

3 zones, 2746 characters of log each


What the commander needs is five fields, and every one is something a decision branches on.

In [2]:
from pydantic import BaseModel, Field, ValidationError


class ZoneReport(BaseModel):
    """The only thing allowed to cross back from a zone worker."""

    zone: str
    robots_migrated: int
    robots_failed: int
    needs_human: bool
    note: str = Field(max_length=160)

Now ask a worker for a summary the way a first version asks. In words.

In [3]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("05-subagent-delegation/03-capstone-a-delegated-migration")

reply = client.chat.completions.create(
    model=model_for("default"), max_tokens=200,
    messages=[{"role": "system",
               "content": "You supervise a warehouse robot fleet migration. Summarise the zone log."},
              {"role": "user", "content": ZONES["Z-DOCK"]}])
prose = (reply.choices[0].message.content or "").strip()
print(f"the worker read {reply.usage.prompt_tokens} tokens and replied with "
      f"{len(prose)} characters\n")
print(prose[:220])

the worker read 1195 tokens and replied with 800 characters

Here's a summary of the ZONE Z-DOCK migration log:

The migration to firmware v5.0.2 for the Z-DOCK zone has been completed. All 41 robots in the zone successfully updated to the new firmware.

**Key observations:**

*  


It is a good summary and a person would act on it. Hand it to the commander's parser, which is the
thing that has to act on it.

In [4]:
# A parser that guesses is worse than one that stops, so let this raise.
report = ZoneReport.model_validate_json(prose)

ValidationError: 1 validation error for ZoneReport
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value="Here's a summary of the ...Overall Status:** While", input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid

## The diagnosis

`ValidationError`, on the first character, because the reply is not JSON at all.

That is the good version of this failure. The dangerous version is a field spelled differently, since
then the commander branches on something nobody declared. A summary is written for a reader. The
commander is a switch.

The worker was asked for a summary and wrote one. The shape was never part of the request, so nothing
in between rejects a wrong answer.

## The fix

The contract has two halves. The first returns a `ZoneReport` or a `ZoneReport`. Never a string,
never `None`.

In [5]:
def validate_report(zone, text):
    """The counter. A filled in form crosses, anything else becomes a flagged one."""
    try:
        return ZoneReport.model_validate_json(text)
    except ValidationError as exc:
        return ZoneReport(zone=zone, robots_migrated=0, robots_failed=0,
                          needs_human=True,
                          note=f"report rejected at the boundary: {type(exc).__name__}")

The second half asks for the right shape in the first place. A structured output puts the fixed shape
on the request itself.

In [6]:
SCHEMA = ZoneReport.model_json_schema() | {"additionalProperties": False}
FORMAT = {"type": "json_schema",
          "json_schema": {"name": "zone_report", "strict": True, "schema": SCHEMA}}
READ_TOKENS = []


def collect_zone(zone, log):
    """One worker. It reads a whole log and returns one validated report."""
    answer = client.chat.completions.create(
        model=model_for("default"), max_tokens=250, response_format=FORMAT,
        messages=[{"role": "system", "content": "You supervise a warehouse robot fleet "
                                                "migration. Report only what the log states."},
                  {"role": "user", "content": log}])
    READ_TOKENS.append(answer.usage.prompt_tokens)
    return validate_report(zone, (answer.choices[0].message.content or "").strip())

Then the parent state, which says what the commander may hold. No channel for a log.

In [7]:
import json
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class MigrationState(TypedDict):
    """Zone names in, validated reports out. Nowhere to put a log."""

    zones: list
    reports: Annotated[list, operator.add]

Wire it the same way. One entry edge, one `Send` per zone, and a worker that returns one channel.

In [8]:
graph = StateGraph(MigrationState)
graph.add_node("fan_out", lambda s: {})
graph.add_node("collect", lambda s: {"reports": [collect_zone(s["zone"], s["log"]).model_dump()]})
graph.add_edge(START, "fan_out")
graph.add_conditional_edges(
    "fan_out",
    lambda s: [Send("collect", {"zone": z, "log": ZONES[z]}) for z in s["zones"]],
    ["collect"])
graph.add_edge("collect", END)

final = graph.compile().invoke({"zones": list(ZONES), "reports": []})
reports = [ZoneReport(**row) for row in final["reports"]]
for report in sorted(reports, key=lambda r: r.zone):
    print(f"  {report.zone:8} {report.robots_migrated:>4} moved  {report.robots_failed:>2} failed"
          f"  needs_human={report.needs_human}")
print(f"\nparent state is {len(json.dumps(final))} characters for "
      f"{sum(len(log) for log in ZONES.values())} characters of log")

  Z-DOCK     41 moved   9 failed  needs_human=True
  Z-NORTH   118 moved   2 failed  needs_human=True
  Z-SOUTH    96 moved   0 failed  needs_human=False

parent state is 498 characters for 8241 characters of log


Every field is typed, so the commander branches without reading. Now the number this vault has been
about. The same decision, from the logs and from the reports.

In [9]:
ROLLUP = "You are the migration commander. Say in one line whether the rollout continues."

with_logs = client.chat.completions.create(
    model=model_for("default"), max_tokens=120,
    messages=[{"role": "system", "content": ROLLUP},
              {"role": "user", "content": "\n\n".join(ZONES.values())}])
with_reports = client.chat.completions.create(
    model=model_for("default"), max_tokens=120,
    messages=[{"role": "system", "content": ROLLUP},
              {"role": "user", "content": json.dumps(final["reports"])}])

print(f"before : {with_logs.usage.prompt_tokens:>5} tokens   "
      f"{(with_logs.choices[0].message.content or '').strip()}")
print(f"after  : {with_reports.usage.prompt_tokens:>5} tokens   "
      f"{(with_reports.choices[0].message.content or '').strip()}")
print(f"\nthe workers read {sum(READ_TOKENS)} tokens the commander never carried")
print(f"zones a person must look at: {[r.zone for r in reports if r.needs_human]}")

before :  3559 tokens   The rollout continues, but monitor Zone Z-Dock closely for further watchdog resets.
after  :   169 tokens   Rollout continues with human intervention required in Z-NORTH and Z-DOCK.

the workers read 3587 tokens the commander never carried
zones a person must look at: ['Z-NORTH', 'Z-DOCK']


## The gate

Two properties, one check. No log line reaches the parent, and the boundary cannot return anything
that is not a report. The prose from the failure is the awkward input.

In [10]:
def test_the_parent_is_never_handed_prose_or_a_log():
    refused = validate_report("Z-DOCK", prose)
    assert isinstance(refused, ZoneReport), "the boundary let a string through"
    assert refused.needs_human, "a rejected report was not flagged for a person"
    body = json.dumps(final["reports"])
    assert "aisle 01 beacon" not in body, "a raw log line is sitting in the parent state"


test_the_parent_is_never_handed_prose_or_a_log()
print("gate holds: prose is refused and flagged, and no log line reached the commander")

gate holds: prose is refused and flagged, and no log line reached the commander


Make `validate_report` return the raw text when validation rejects it, and the first line fails.

### Enterprise exploration

- The fallback says zero moved and zero failed. What does the commander do with a zone it has no
  numbers for, and is a flagged zero safer than no row?
- Thirty zones instead of three. What does that fan out cost, and where do you cap it?
- A worker reported numbers the log does not contain. Nothing here catches that. What check would, and
  what does it cost per zone?
- Migration records are retained for safety audits, and the commander only saw reports. Where do the
  logs live, and who can tie a report back to one?

### Key takeaways

- A boundary returns one type, or it has declared nothing.
- `response_format` shapes the reply. Pydantic rejects it if the shape is still wrong. Both, not one.
- A rejected report becomes a flagged report, so the parent never has to read anything.
- The parent carried a fraction of the logs, and the workers still read every line.